# 第 15 课：流式特征提取——Chunk、帧边界与缓存

目标：把任意大小的音频块送入系统，同时产生与离线分帧一致的帧，不丢样本、不重复计算。

<!-- course-upgrade-v2 -->
## 学习导航

| 项目 | 内容 |
|---|---|
| 所属阶段 | 流式 ASR |
| 建议投入 | 3～5 小时，可分 2～3 次完成 |
| 前置要求 | 完成第 14 课；如果前测低于 2/3，先回看上一课小结 |
| 本课核心 | 音频 chunk、跨块缓存、在线分帧 |
| 完成标准 | 能口头解释核心概念；独立完成强化题；从空白重写核心函数 |

高效顺序：**先回答前测 → 预测代码结果 → 再运行 → 修改一个变量 → 关闭答案复现 → 次日回忆。**


<!-- course-upgrade-v2 -->
## 课前诊断（先不要运行代码）

1. 分别用一句话解释：音频 chunk、跨块缓存、在线分帧。
2. 画出这三个概念之间的输入—输出关系。
3. 写下你最不确定的一点，并给出一个暂时猜测。

自评：答对 0～1 题先复习前置课；答对 2 题可以正常学习；3 题都能讲清楚则直接挑战代码和迁移题。


In [1]:
from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt

def find_root():
    here = Path.cwd().resolve()
    for p in [here, *here.parents]:
        if (p / "pyproject.toml").exists(): return p
    raise FileNotFoundError("请从 learn_asr 或 notebooks 目录启动 Jupyter")

ROOT = find_root()
BLANK = "∅"
plt.rcParams["figure.figsize"] = (11, 4)
print("项目根目录:", ROOT)

import soundfile as sf
from ipywidgets import interact, IntSlider
y,sr=sf.read(ROOT/"data"/"spoken_digits_parts"/"3_jackson_0.wav")
y=y.astype(np.float32); frame_length=round(.025*sr); hop=round(.010*sr)
print(sr,frame_length,hop,len(y))

项目根目录: <REPO_ROOT>
8000 200 80 3886


## 1. 离线分帧作为标准答案

In [2]:
def offline_frames(x,L,H):
    if len(x)<L:return np.empty((0,L),x.dtype)
    n=1+(len(x)-L)//H
    return np.stack([x[i*H:i*H+L] for i in range(n)])
reference=offline_frames(y,frame_length,hop)
print("离线帧数",len(reference))

离线帧数 47


## 2. 有状态 StreamingFramer

In [3]:
class StreamingFramer:
    def __init__(self,L,H): self.L,self.H=L,H; self.buffer=np.empty(0,np.float32)
    def accept(self,chunk):
        self.buffer=np.concatenate([self.buffer,np.asarray(chunk,dtype=np.float32)])
        out=[]
        while len(self.buffer)>=self.L:
            out.append(self.buffer[:self.L].copy()); self.buffer=self.buffer[self.H:]
        return np.stack(out) if out else np.empty((0,self.L),np.float32)

for chunk_ms in [7,20,100,333]:
    size=max(1,round(sr*chunk_ms/1000)); fr=StreamingFramer(frame_length,hop);parts=[]
    for start in range(0,len(y),size): parts.append(fr.accept(y[start:start+size]))
    got=np.concatenate(parts) if parts else np.empty_like(reference)
    print(f"chunk={chunk_ms:3d} ms frames={len(got):3d} max_error={np.max(np.abs(got-reference)):.1e}")

chunk=  7 ms frames= 47 max_error=0.0e+00
chunk= 20 ms frames= 47 max_error=0.0e+00
chunk=100 ms frames= 47 max_error=0.0e+00
chunk=333 ms frames= 47 max_error=0.0e+00


## 3. 为什么必须缓存

25 ms 窗、10 ms hop 意味着相邻帧重叠 15 ms。chunk 边界经常切在一帧中间；直接对每块单独做 STFT 会丢掉跨边界帧。

In [4]:
@interact(chunk_ms=IntSlider(min=5,max=100,value=30,step=5,description="chunk ms"))
def draw_boundaries(chunk_ms=30):
    duration=len(y)/sr; starts=np.arange(0,duration,hop/sr); chunks=np.arange(0,duration,chunk_ms/1000)
    fig,ax=plt.subplots(figsize=(11,2.4))
    for s in starts[:35]: ax.plot([s,s+frame_length/sr],[.55,.55],color="C0",alpha=.35)
    for c in chunks: ax.axvline(c,color="C1",alpha=.8)
    ax.set(xlim=(0,min(duration,.4)),ylim=(0,1),yticks=[],xlabel="Time (s)",title="Blue frame spans; orange chunk boundaries")
    plt.show()

interactive(children=(IntSlider(value=30, description='chunk ms', min=5, step=5), Output()), _dom_classes=('wi…

## 4. 在线特征还要注意

- `center=False` 更容易定义真实时间边界；
- 全句均值方差归一化偷看未来，不严格在线；
- 在线 CMVN 需要累计统计量或固定训练集统计量；
- 最后不足一帧的尾部要规定丢弃还是补零；
- 音频采集时钟、重采样器也可能有状态。

## 本课测试

1. 25 ms window、10 ms hop 的重叠是多少？
2. chunk=20 ms 时为什么仍可能暂时产不出第一帧？
3. chunk 大小必须是 hop 的整数倍吗？
4. 全句 CMVN 为什么不流式？
5. 缓存长度是否永远固定为 `window-hop`？

<details><summary>展开参考答案</summary>

1. 15 ms。2. 第一帧需要收满 25 ms。3. 不必，有状态缓冲器可以处理任意大小。4. 它需要未来整句统计量。5. 简单分帧器消费后通常保留未消费尾部，长度会变化但小于 window；其他前端模块的状态另算。

</details>

<!-- course-upgrade-v2 -->
## 强化练习：第 15 课专属题库

请先把答案写进新的 Markdown/Code cell，再展开自评标准。

### A. 基础回忆

1. 不看上文，分别定义 `音频 chunk`、`跨块缓存`、`在线分帧`。
2. 哪一个量/状态是本课最容易在模块边界丢失的？它的单位和 shape 是什么？
3. 本课至少写出两个“看起来能运行，但结果其实错误”的例子。

### B. 预测与推理

4. 场景：**chunk 边界落在一帧中间**。先预测现象，再说明原因，最后给出一项可以验证猜测的指标。
5. 改变本课最关键参数的 0.5×、1×、2×，分别预测准确率、延迟、内存或数值误差怎样变化。
6. 画一张最小数据流图，在每条边标出 dtype、shape、时间单位或概率/代价方向。

### C. 编程与排错

7. 编程任务：**用不规则 chunk 验证离线/在线帧完全一致**。至少加入正常、边界、错误输入三类测试。
8. 故意制造一个 off-by-one、shape、状态未 reset 或数值稳定性错误；记录错误现象和定位过程。
9. 不看本课实现，从空白 cell 重写最核心函数，并用原实现作数值对照。

### D. 迁移与表达

10. 跨课任务：**把前端 cache 与 encoder cache 区分**。
11. 用 90 秒向没有学过 ASR 的人解释本课；禁止只念术语，必须举一个数字或生活例子。
12. 写出一个生产系统中会监控的指标，以及它异常时优先检查的三处位置。

<details><summary>展开自评标准</summary>

- 每题 0～2 分：0=无法回答；1=方向正确但缺少单位、边界或验证；2=解释完整且能用代码/数字验证。
- 24 分满分：达到 19 分再进入下一课；15～18 分次日重做错题；低于 15 分回看本课图和核心代码。
- 第 4 题必须包含“预测—原因—指标”，第 7～9 题必须真正运行测试，第 10 题必须明确上下游 contract。
- 核心答案至少应正确使用：音频 chunk、跨块缓存、在线分帧。

</details>


<!-- course-upgrade-v2 -->
## 间隔复习与离场票

### 离场票（现在完成）

- [ ] 我能不用笔记解释 音频 chunk、跨块缓存、在线分帧。
- [ ] 我能说出本课最常见的错误及其观测现象。
- [ ] 我能从空白重写一个核心函数，并通过至少 3 个测试。
- [ ] 我能说明本课对上一层和下一层接口的影响。

### 复习时间表

- **明天（5 分钟）**：闭卷写出三个核心概念和一个公式/shape。
- **7 天后（15 分钟）**：重做第 4、7、10 题，不运行原答案。
- **30 天后（20 分钟）**：从真实音频或随机张量重新构造一个最小实验。

把错题记录到根目录 `LEARNING_LOG.md`。不要只写“不会”，要写：原判断、证据、正确规则、下次检查动作。
